In [ ]:
# ============================================================
# Explainable AI for Credit Risk: Taiwan Dataset
# ============================================================
#
# This notebook contains the Taiwan dataset analysis used in the MSc
# dissertation "Explainable AI for Credit Risk Assessment".
#
# The workflow covers data preparation, four predictive models, SHAP, LIME
# and Integrated Gradients explanations, demographic outcome analysis, and
# the TensorFlow/PyTorch reproducibility comparison.
#
# Categorical feature attributions are aggregated back to their original
# parent features before ranking so that explanation methods can be compared
# on a common feature representation. Helper functions are also used to
# standardise the different output structures returned by the SHAP explainers.
#
# For the TensorFlow/PyTorch comparison, common data splits and fixed
# explanation samples are used to keep the two implementations as closely
# matched as practicable. Repeated executions may nevertheless produce small
# numerical differences because complete determinism is not guaranteed,
# particularly for TensorFlow and stochastic LIME sampling.
#
# Integrated Gradients is implemented with Captum for PyTorch and directly
# with TensorFlow automatic differentiation using the method described by
# Sundararajan, Taly and Yan (2017).


In [ ]:
# ============================================================
# Setup (Google Colab)
# ============================================================

# Google Colab provides the core scientific Python and machine-learning libraries.
# Install the additional packages required for dataset access and XAI methods.
!pip install -q ucimlrepo
!pip install -q shap lime captum

# --- Standard Library ---
import os, time, copy

# --- Core Numerics and Data ---
import numpy as np
import pandas as pd

# --- Dataset Source ---
from ucimlrepo import fetch_ucirepo

# --- Models ---
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# --- Preprocessing, Model Selection, Metrics ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# --- Explanation Methods ---
import shap
import lime.lime_tabular
from captum.attr import IntegratedGradients

# --- Statistical Tests ---
from scipy.stats import spearmanr, wilcoxon

# --- Colab File Download ---
from google.colab import files

# --- Record Key Library Versions ---
# Record the main package versions used in the Colab environment
import sklearn, xgboost
print("tensorflow", tf.__version__)
print("torch", torch.__version__)
print("sklearn", sklearn.__version__)
print("xgboost", xgboost.__version__)

In [ ]:
# ============================================================
# Load the Taiwan Data
# ============================================================
# Data: Yeh (2009), Default of Credit Card Clients, UCI Machine Learning Repository
# Originating study: Yeh and Lien (2009)

# Pull straight from UCI by id (350 is the default of credit card clients set)
default_taiwan = fetch_ucirepo(id=350)

# Use dataset-specific variable names to keep the Taiwan and German analyses distinct
X_taiwan = default_taiwan.data.features
y_taiwan = default_taiwan.data.targets

# Columns arrive as X1..X23, so swap in the readable names from the variable table
readable = default_taiwan.variables.set_index('name')['description'].to_dict()
X_taiwan = X_taiwan.rename(columns=readable)

X_taiwan.columns.tolist()

In [ ]:
# ============================================================
# Clean EDUCATION
# ============================================================

# Recode undocumented values 0, 5 and 6 to category 4 (Other)
X_taiwan['EDUCATION'] = X_taiwan['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

X_taiwan['EDUCATION'].value_counts().sort_index()

In [ ]:
# ============================================================
# Clean MARRIAGE
# ============================================================

# Recode undocumented value 0 to category 3 (Other)
X_taiwan['MARRIAGE'] = X_taiwan['MARRIAGE'].replace({0: 3})

X_taiwan['MARRIAGE'].value_counts().sort_index()

In [ ]:
# ============================================================
# Repayment Columns (PAY)
# ============================================================

# PAY_0 is actually the latest month, rename it PAY_1 so the six months read in order
X_taiwan = X_taiwan.rename(columns={'PAY_0': 'PAY_1'})

# Retain the repayment-status values as an ordered variable:
# -2 = no credit used, -1 = paid in full, 0 = revolving/minimum payment,
# and 1-8 = number of months overdue
[c for c in X_taiwan.columns if c.startswith('PAY_')]

In [ ]:
# ============================================================
# Train / Test Split
# ============================================================

# Convert the single-column target frame to a Series for scikit-learn compatibility
y_taiwan = y_taiwan.squeeze()

# Split before scaling to prevent information from the test set entering preprocessing
# Stratify to preserve the approximately 22% default rate, using a fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X_taiwan, y_taiwan,
    test_size=0.2,
    stratify=y_taiwan,
    random_state=42,
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# Encode and Scale (Fit on Train Only)
# ============================================================

# Work on explicit copies of the training and test sets
X_train = X_train.copy()
X_test = X_test.copy()

# Recode sex as a binary indicator: 0 = male, 1 = female
X_train['SEX'] = X_train['SEX'].map({1: 0, 2: 1})
X_test['SEX']  = X_test['SEX'].map({1: 0, 2: 1})

# One-hot encode the nominal education and marriage variables, dropping the reference level
X_train = pd.get_dummies(
    X_train,
    columns=['EDUCATION', 'MARRIAGE'],
    drop_first=True,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=['EDUCATION', 'MARRIAGE'],
    drop_first=True,
    dtype=int
)

# Align test-set columns with the training set in case a category is absent from one split
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Standardise numeric and ordinal features using parameters fitted on the training set only
numeric_cols = [
    'LIMIT_BAL', 'AGE',
    'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train.shape, X_test.shape)
X_train.head()

In [ ]:
# ============================================================
# Information Value (Diagnostic, No Features Dropped)
# ============================================================

# Score the un-encoded training rows so categoricals count as one variable, not dummies
X_iv = X_taiwan.loc[X_train.index].copy()
y_iv = y_train

def iv_for_feature(x, y, n_bins=10):
    # Deciles for anything with lots of distinct values, otherwise the values are their own bins
    if x.nunique() > 20:
        binned = pd.qcut(x, n_bins, duplicates='drop')
    else:
        binned = x.astype('category')

    d = pd.DataFrame({'bin': binned, 'y': y.values})
    grp = d.groupby('bin', observed=True)['y'].agg(['count', 'sum'])
    grp.columns = ['total', 'bad']          # bad = defaults (y = 1)
    grp['good'] = grp['total'] - grp['bad']

    n = len(grp)
    grp['dist_bad']  = (grp['bad']  + 0.5) / (grp['bad'].sum()  + 0.5 * n)
    grp['dist_good'] = (grp['good'] + 0.5) / (grp['good'].sum() + 0.5 * n)

    grp['woe'] = np.log(grp['dist_good'] / grp['dist_bad'])
    return ((grp['dist_good'] - grp['dist_bad']) * grp['woe']).sum()

iv_table = pd.Series(
    {col: iv_for_feature(X_iv[col], y_iv) for col in X_iv.columns}
).sort_values(ascending=False).round(4)

print(iv_table)

In [ ]:
# ============================================================
# Model 1: Logistic Regression
# ============================================================
# Class weighting is used in place of synthetic resampling (Chawla et al., 2002);
# See Section 3.3 of the report for why fabricated instances are unsuitable here

# Class_weight balanced handles the 22% default rate without resampling
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)

proba = logreg.predict_proba(X_test)[:, 1]
print("Test AUC:", round(roc_auc_score(y_test, proba), 4))
print()
print(classification_report(y_test, logreg.predict(X_test)))

# Use the same fixed cross-validation folds across all models for consistent comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(logreg, X_train, y_train, cv=cv, scoring='roc_auc')
print("CV AUC (5-fold):", cv_auc.round(4))
print("Mean CV AUC:", round(cv_auc.mean(), 4))

In [ ]:
# ============================================================
# Model 2: XGBoost
# ============================================================
# XGBoost (Chen and Guestrin, 2016)

# Scale_pos_weight does the same job as class_weight, ratio of non-defaults to defaults
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)

proba = xgb.predict_proba(X_test)[:, 1]
print("Test AUC:", round(roc_auc_score(y_test, proba), 4))
print()
print(classification_report(y_test, xgb.predict(X_test)))

xgb_cv_auc = cross_val_score(xgb, X_train, y_train, cv=cv, scoring='roc_auc')
print("CV AUC (5-fold):", xgb_cv_auc.round(4))
print("Mean CV AUC:", round(xgb_cv_auc.mean(), 4))

# Store both models' fold scores for the later comparison
cv_results = {'Logistic Regression': cv_auc, 'XGBoost': xgb_cv_auc}

In [ ]:
# ============================================================
# Model 3: Neural Network (TensorFlow / Keras)
# ============================================================
# Adam optimiser (Kingma and Ba, 2015)

# Set random seeds to reduce avoidable stochastic variation
tf.random.set_seed(42)
np.random.seed(42)

# Use balanced class weights to handle class imbalance consistently across models
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: weights[0], 1: weights[1]}

# Float arrays for the net
X_train_arr = X_train.astype('float32').values
X_test_arr  = X_test.astype('float32').values
y_train_arr = y_train.astype('float32').values
y_test_arr  = y_test.astype('float32').values

# 26 -> 32 -> 16 -> 1, this exact shape gets mirrored in PyTorch next
tf_model = keras.Sequential([
    keras.layers.Input(shape=(X_train_arr.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')],
)

history = tf_model.fit(
    X_train_arr, y_train_arr,
    validation_split=0.2,
    epochs=50,
    batch_size=128,
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')],
    verbose=0,
)

proba = tf_model.predict(X_test_arr, verbose=0).ravel()
print("Test AUC:", round(roc_auc_score(y_test_arr, proba), 4))
print()
print(classification_report(y_test_arr, (proba >= 0.5).astype(int)))
print("Epochs trained:", len(history.history['loss']))

In [ ]:
# ============================================================
# Model 4: Neural Network (PyTorch)
# ============================================================

# Set random seeds to reduce avoidable stochastic variation
torch.manual_seed(42)
np.random.seed(42)

# Same last-20% validation split Keras uses
n_val = int(0.2 * len(X_train_arr))
X_tr, X_val = X_train_arr[:-n_val], X_train_arr[-n_val:]
y_tr, y_val = y_train_arr[:-n_val], y_train_arr[-n_val:]

X_tr_t   = torch.tensor(X_tr,  dtype=torch.float32)
y_tr_t   = torch.tensor(y_tr,  dtype=torch.float32).view(-1, 1)
X_val_t  = torch.tensor(X_val, dtype=torch.float32)
y_val_t  = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_arr, dtype=torch.float32)

# Same shape as the TF net: 26 -> 32 -> 16 -> 1
class Net(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)

torch_model = Net(X_tr_t.shape[1])
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.001)

# Per-sample class weights, mirroring Keras class_weight
w0, w1 = float(class_weight[0]), float(class_weight[1])
def weighted_bce(pred, target):
    w = torch.where(target == 1, torch.tensor(w1), torch.tensor(w0))
    return F.binary_cross_entropy(pred, target, weight=w)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)

# Training loop with early stopping on val loss, patience 8, restore best weights
best_val, best_state, patience, wait, epochs_run = float('inf'), None, 8, 0, 0
for epoch in range(50):
    torch_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = weighted_bce(torch_model(xb), yb)
        loss.backward()
        optimizer.step()

    torch_model.eval()
    with torch.no_grad():
        val_loss = weighted_bce(torch_model(X_val_t), y_val_t).item()

    epochs_run += 1
    if val_loss < best_val:
        best_val, wait = val_loss, 0
        best_state = copy.deepcopy(torch_model.state_dict())
    else:
        wait += 1
        if wait >= patience:
            break

torch_model.load_state_dict(best_state)

# Test performance
torch_model.eval()
with torch.no_grad():
    proba = torch_model(X_test_t).numpy().ravel()

print("Test AUC:", round(roc_auc_score(y_test_arr, proba), 4))
print()
print(classification_report(y_test_arr, (proba >= 0.5).astype(int)))
print("Epochs trained:", epochs_run)

In [ ]:
# ============================================================
# Reusable Net Training Functions (for CV and Later RQ2 Seeds)
# ============================================================
# Adam optimiser (Kingma and Ba, 2015)

def train_tf_net(X_tr, y_tr, seed=42, epochs=50):
    tf.random.set_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    model = keras.Sequential([
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    model.fit(X_tr, y_tr, validation_split=0.2, epochs=epochs, batch_size=128,
              class_weight={0: cw[0], 1: cw[1]},
              callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
              verbose=0)
    return model

def train_torch_net(X_tr, y_tr, seed=42, epochs=50):
    torch.manual_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    w0, w1 = float(cw[0]), float(cw[1])

    n_val = int(0.2 * len(X_tr))
    Xt, Xv = X_tr[:-n_val], X_tr[-n_val:]
    yt, yv = y_tr[:-n_val], y_tr[-n_val:]
    Xt_t = torch.tensor(Xt, dtype=torch.float32); yt_t = torch.tensor(yt, dtype=torch.float32).view(-1, 1)
    Xv_t = torch.tensor(Xv, dtype=torch.float32); yv_t = torch.tensor(yv, dtype=torch.float32).view(-1, 1)

    model = nn.Sequential(
        nn.Linear(X_tr.shape[1], 32), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    def wbce(p, t):
        w = torch.where(t == 1, torch.tensor(w1), torch.tensor(w0))
        return F.binary_cross_entropy(p, t, weight=w)

    loader = DataLoader(TensorDataset(Xt_t, yt_t), batch_size=128, shuffle=True)
    best, best_state, wait = float('inf'), None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss = wbce(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = wbce(model(Xv_t), yv_t).item()
        if vl < best:
            best, wait, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= 8:
                break
    model.load_state_dict(best_state)
    return model

In [ ]:
# ============================================================
# Cross-Validate the Nets (Matched Folds for the Comparison)
# ============================================================

tf_cv_auc, torch_cv_auc = [], []
for tr_idx, va_idx in cv.split(X_train, y_train):
    Xtr = X_train.iloc[tr_idx].astype('float32').values
    ytr = y_train.iloc[tr_idx].astype('float32').values
    Xva = X_train.iloc[va_idx].astype('float32').values
    yva = y_train.iloc[va_idx].astype('float32').values

    m_tf = train_tf_net(Xtr, ytr)
    tf_cv_auc.append(roc_auc_score(yva, m_tf.predict(Xva, verbose=0).ravel()))

    m_pt = train_torch_net(Xtr, ytr)
    with torch.no_grad():
        p = m_pt(torch.tensor(Xva, dtype=torch.float32)).numpy().ravel()
    torch_cv_auc.append(roc_auc_score(yva, p))

tf_cv_auc, torch_cv_auc = np.array(tf_cv_auc), np.array(torch_cv_auc)
print("TF CV AUC:   ", tf_cv_auc.round(4), "mean", round(tf_cv_auc.mean(), 4))
print("Torch CV AUC:", torch_cv_auc.round(4), "mean", round(torch_cv_auc.mean(), 4))

# Now all four models have matched fold scores for the Friedman test
cv_results['TensorFlow NN'] = tf_cv_auc
cv_results['PyTorch NN'] = torch_cv_auc

In [ ]:
# ============================================================
# Model Comparison Summary
# ============================================================

# Per-fold AUCs for all four models in one frame
cv_df = pd.DataFrame(cv_results)            # rows = folds, columns = models

summary = pd.DataFrame({
    'mean_auc': cv_df.mean(),
    'std_auc':  cv_df.std(),
}).sort_values('mean_auc', ascending=False).round(4)

print(summary)
print()
print("Per-fold AUCs (this matrix is what the Friedman / Nemenyi test uses in R):")
print(cv_df.round(4))

In [ ]:
# ============================================================
# RQ3 Fairness: Four-Fifths Rule on Test Predictions
# ============================================================
# Four-fifths rule (Equal Employment Opportunity Commission, 1978); bootstrap
# Confidence intervals for the ratios are computed in R (Efron, 1979)

# Original, un-encoded demographics for the test rows
demo = X_taiwan.loc[X_test.index].copy()
demo['SEX_label'] = demo['SEX'].map({1: 'male', 2: 'female'})
demo['AGE_band'] = pd.cut(
    demo['AGE'],
    bins=[20, 30, 40, 50, 60, 100],
    labels=['21-30', '31-40', '41-50', '51-60', '61+']
)

# Make sure the torch net is in eval mode before predicting
torch_model.eval()

# Predicted labels from each model (1 = predicted default, 0 = predicted non-default)
preds = {
    'Logistic Regression': logreg.predict(X_test),
    'XGBoost': xgb.predict(X_test),
    'TensorFlow NN': (tf_model.predict(X_test_arr, verbose=0).ravel() >= 0.5).astype(int),
    'PyTorch NN': (torch_model(torch.tensor(X_test_arr, dtype=torch.float32)).detach().numpy().ravel() >= 0.5).astype(int),
}

def four_fifths(pred, group):
    # Favourable outcome = predicted NOT to default (pred == 0)
    rates = pd.Series(pred == 0).groupby(group.values).mean()
    return rates.round(3), round(rates.min() / rates.max(), 3)

for name, pred in preds.items():
    print(name)
    for attr in ['SEX_label', 'AGE_band']:
        rates, di = four_fifths(pred, demo[attr])
        flag = "   <-- below 0.8 screening threshold" if di < 0.8 else ""
        print(f"  {attr}: disparate impact ratio = {di}{flag}")
        print("   ", rates.to_dict())
    print()

In [ ]:
# ============================================================
# Export Results for the R Analysis
# ============================================================

# Per-fold AUCs for the Friedman / Nemenyi test in R
cv_df.to_csv('taiwan_cv_auc.csv', index_label='fold')

# Test predictions, truth and demographics for any R-side checks
export = demo[['SEX', 'AGE', 'MARRIAGE']].copy()
export['y_true'] = y_test.values
for name, pred in preds.items():
    export[name.replace(' ', '_')] = pred
export.to_csv('taiwan_test_predictions.csv', index=False)

print("Saved taiwan_cv_auc.csv and taiwan_test_predictions.csv")

# Download the exported CSV files from the Colab session
from google.colab import files
files.download('taiwan_cv_auc.csv')
files.download('taiwan_test_predictions.csv')

In [ ]:
# ============================================================
# Explanation Setup: Shared Helpers and the Parent-Feature Map
# ============================================================

feature_names = X_train.columns.tolist()

# Fold the education and marriage dummies back to one feature each,
# Everything else (Pay_1, bill_amt1, etc) maps to itself
def to_parent(col):
    if col.startswith('EDUCATION_'):
        return 'EDUCATION'
    if col.startswith('MARRIAGE_'):
        return 'MARRIAGE'
    return col

parent_of = {c: to_parent(c) for c in feature_names}

# Normalise whatever shape a SHAP explainer returns to one (N, p) array for the positive class
def as_2d(sv):
    if isinstance(sv, list):
        sv = sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, -1]
    return sv

# Global importance = mean absolute attribution per feature, summed back to parent features
def global_importance(sv_2d):
    s = pd.Series(np.abs(sv_2d).mean(axis=0), index=feature_names)
    return s.groupby(s.index.map(parent_of)).sum().sort_values(ascending=False)

# Store the model and explanation-method importance vectors for the concordance analysis
importance_store = {}

In [ ]:
# ============================================================
# SHAP: Logistic Regression (LinearExplainer) and XGBoost (TreeExplainer)
# ============================================================
# SHAP (Lundberg and Lee, 2017); the exact tree explainer is Lundberg et al. (2020)

# Logistic regression: linear explainer against a training background sample
background = X_train.sample(n=1000, random_state=42)
lin_sv = as_2d(shap.LinearExplainer(logreg, background).shap_values(X_test))
importance_store[('Logistic Regression', 'SHAP')] = global_importance(lin_sv)

# XGBoost: use TreeExplainer; no background sample is required
tree_sv = as_2d(shap.TreeExplainer(xgb).shap_values(X_test))
importance_store[('XGBoost', 'SHAP')] = global_importance(tree_sv)

for name in ['Logistic Regression', 'XGBoost']:
    print(name, "- SHAP top 10 features")
    print(importance_store[(name, 'SHAP')].head(10).round(4))
    print()

In [ ]:
# ============================================================
# SHAP: The Two Neural Nets (GradientExplainer)
# ============================================================
# Gradient explainer chosen for the networks: shares a gradient basis with
# Integrated Gradients (Sundararajan, Taly and Yan, 2017), so the two are comparable

# Reuse the same fixed test sample across SHAP, LIME and IG for consistent comparison
rng = np.random.RandomState(42)
explain_idx = rng.choice(X_test_arr.shape[0], size=500, replace=False)
X_explain_arr = X_test_arr[explain_idx]

# Small background sample for the gradient explainers
bg_idx = rng.choice(X_train_arr.shape[0], size=200, replace=False)
X_bg_arr = X_train_arr[bg_idx]

# Tensorflow net
tf_sv = as_2d(shap.GradientExplainer(tf_model, X_bg_arr).shap_values(X_explain_arr))
importance_store[('TensorFlow NN', 'SHAP')] = global_importance(tf_sv)

# Pytorch net, eval mode and tensors
torch_model.eval()
torch_sv = as_2d(
    shap.GradientExplainer(torch_model, torch.tensor(X_bg_arr, dtype=torch.float32))
    .shap_values(torch.tensor(X_explain_arr, dtype=torch.float32))
)
importance_store[('PyTorch NN', 'SHAP')] = global_importance(torch_sv)

for name in ['TensorFlow NN', 'PyTorch NN']:
    print(name, "- SHAP top 10 features")
    print(importance_store[(name, 'SHAP')].head(10).round(4))
    print()

In [ ]:
# ============================================================
# Integrated Gradients on the Two Nets (Reference Baseline)
# ============================================================
# Integrated Gradients (Sundararajan, Taly and Yan, 2017)
# The zero vector represents the training mean for standardised numeric features
# and the inactive/reference level for encoded categorical features

# Pytorch via captum
torch_model.eval()
X_explain_t = torch.tensor(X_explain_arr, dtype=torch.float32)
ig_attr = IntegratedGradients(torch_model).attribute(
    X_explain_t, baselines=torch.zeros_like(X_explain_t), n_steps=64)
importance_store[('PyTorch NN', 'IG')] = global_importance(ig_attr.detach().numpy())

# Tensorflow via a manual riemann-sum integrated gradients
def tf_integrated_gradients(model, inputs, steps=64):
    inputs = tf.convert_to_tensor(inputs, dtype=tf.float32)
    baseline = tf.zeros_like(inputs)
    total = tf.zeros_like(inputs)
    for a in tf.linspace(0.0, 1.0, steps + 1):
        x = baseline + a * (inputs - baseline)
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
        total += tape.gradient(preds, x)
    avg_grads = total / tf.cast(steps + 1, tf.float32)
    return ((inputs - baseline) * avg_grads).numpy()

tf_ig = tf_integrated_gradients(tf_model, X_explain_arr)
importance_store[('TensorFlow NN', 'IG')] = global_importance(tf_ig)

for name in ['TensorFlow NN', 'PyTorch NN']:
    print(name, "- IG top 10 features")
    print(importance_store[(name, 'IG')].head(10).round(4))
    print()

In [ ]:
# ============================================================
# LIME on All Four Models (Global Importance from Local Weights)
# ============================================================
# LIME (Ribeiro, Singh and Guestrin, 2016)

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_arr,
    feature_names=feature_names,
    class_names=['no default', 'default'],
    mode='classification',
    discretize_continuous=False,
    random_state=42,
)

# Reuse the first 150 rows of the shared explanation sample to limit computational cost
X_lime = X_explain_arr[:150]

# Define prediction-probability wrappers with a consistent two-column output format
def lr_pp(x):  return logreg.predict_proba(pd.DataFrame(x, columns=feature_names))
def xgb_pp(x): return xgb.predict_proba(pd.DataFrame(x, columns=feature_names))
def tf_pp(x):
    p = tf_model(x.astype('float32'), training=False).numpy().ravel()
    return np.column_stack([1 - p, p])
def torch_pp(x):
    with torch.no_grad():
        p = torch_model(torch.tensor(x, dtype=torch.float32)).numpy().ravel()
    return np.column_stack([1 - p, p])

def lime_global(predict_fn):
    acc = np.zeros(len(feature_names))
    for row in X_lime:
        exp = lime_explainer.explain_instance(
            row, predict_fn, num_features=len(feature_names), labels=(1,), num_samples=1000)
        for idx, w in exp.as_map()[1]:
            acc[idx] += abs(w)
    s = pd.Series(acc / len(X_lime), index=feature_names)
    return s.groupby(s.index.map(parent_of)).sum().sort_values(ascending=False)

for name, fn in [('Logistic Regression', lr_pp), ('XGBoost', xgb_pp),
                 ('TensorFlow NN', tf_pp), ('PyTorch NN', torch_pp)]:
    print("running lime for", name, "...")
    importance_store[(name, 'LIME')] = lime_global(fn)

print()
for name in ['Logistic Regression', 'XGBoost', 'TensorFlow NN', 'PyTorch NN']:
    print(name, "- LIME top 10 features")
    print(importance_store[(name, 'LIME')].head(10).round(4))
    print()

In [ ]:
# ============================================================
# RQ1 Concordance: Kendall's W and Pairwise Spearman
# ============================================================
# Kendall's coefficient of concordance (Kendall and Babington Smith, 1939)

# Rows = 23 parent features, columns = (Model, method) mean|attribution|
imp_df = pd.DataFrame(importance_store).fillna(0.0)
imp_df.columns = pd.MultiIndex.from_tuples(imp_df.columns, names=['model', 'method'])

# Ranks within each column, 1 = most important
ranks = imp_df.rank(ascending=False)

def kendalls_w(rank_matrix):
    m, n = rank_matrix.shape[1], rank_matrix.shape[0]
    Rj = rank_matrix.sum(axis=1)
    S = ((Rj - Rj.mean()) ** 2).sum()
    return 12 * S / (m ** 2 * (n ** 3 - n))

print("Kendall's W, three-way (SHAP / IG / LIME):")
for net in ['TensorFlow NN', 'PyTorch NN']:
    cols = [(net, m) for m in ['SHAP', 'IG', 'LIME']]
    print(f"  {net}: {kendalls_w(ranks[cols]):.3f}")
print()

print("Pairwise Spearman between methods:")
for model in ['Logistic Regression', 'XGBoost', 'TensorFlow NN', 'PyTorch NN']:
    methods = [m for (mod, m) in imp_df.columns if mod == model]
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            r, _ = spearmanr(imp_df[(model, methods[i])], imp_df[(model, methods[j])])
            print(f"  {model}: {methods[i]} vs {methods[j]} = {r:.3f}")
print()

# Export for the formal R analysis
flat = imp_df.copy()
flat.columns = [f"{mod}__{meth}" for (mod, meth) in flat.columns]
flat.to_csv('taiwan_xai_importance.csv', index_label='feature')
print("Saved taiwan_xai_importance.csv")

In [ ]:
# ============================================================
# RQ2 Setup: Helpers, Instrumented Training Functions, Seeds
# ============================================================
# Adam optimiser (Kingma and Ba, 2015)

DATASET = 'taiwan'

# Map one-hot encoded variables back to their original parent features
feature_names = X_train.columns.tolist()
def to_parent(col):
    if col.startswith('EDUCATION_'): return 'EDUCATION'
    if col.startswith('MARRIAGE_'):  return 'MARRIAGE'
    return col
parent_of = {c: to_parent(c) for c in feature_names}
def as_2d(sv):
    if isinstance(sv, list): sv = sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3: sv = sv[:, :, -1]
    return sv
def global_importance(sv_2d):
    s = pd.Series(np.abs(sv_2d).mean(axis=0), index=feature_names)
    return s.groupby(s.index.map(parent_of)).sum().sort_values(ascending=False)

def train_tf_seed(X_tr, y_tr, seed, epochs=50):
    keras.utils.set_random_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    model = keras.Sequential([
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(32, activation='relu'), keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation='relu'), keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    es = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')
    t0 = time.perf_counter()
    h = model.fit(X_tr, y_tr, validation_split=0.2, epochs=epochs, batch_size=128,
                  class_weight={0: cw[0], 1: cw[1]}, callbacks=[es], verbose=0)
    return model, {'epochs': len(h.history['loss']),
                   'final_val_loss': min(h.history['val_loss']),
                   'train_seconds': time.perf_counter() - t0}

def train_torch_seed(X_tr, y_tr, seed, epochs=50):
    torch.manual_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    w0, w1 = float(cw[0]), float(cw[1])
    n_val = int(0.2 * len(X_tr))
    Xt, Xv = X_tr[:-n_val], X_tr[-n_val:]
    yt, yv = y_tr[:-n_val], y_tr[-n_val:]
    Xt_t = torch.tensor(Xt, dtype=torch.float32); yt_t = torch.tensor(yt, dtype=torch.float32).view(-1, 1)
    Xv_t = torch.tensor(Xv, dtype=torch.float32); yv_t = torch.tensor(yv, dtype=torch.float32).view(-1, 1)
    model = nn.Sequential(nn.Linear(X_tr.shape[1], 32), nn.ReLU(), nn.Dropout(0.3),
                          nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    def wbce(p, t):
        w = torch.where(t == 1, torch.tensor(w1), torch.tensor(w0))
        return F.binary_cross_entropy(p, t, weight=w)
    loader = DataLoader(TensorDataset(Xt_t, yt_t), batch_size=128, shuffle=True)
    best, best_state, wait, ep = float('inf'), None, 0, 0
    t0 = time.perf_counter()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); wbce(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = wbce(model(Xv_t), yv_t).item()
        ep += 1
        if vl < best: best, wait, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= 8: break
    model.load_state_dict(best_state)
    return model, {'epochs': ep, 'final_val_loss': best, 'train_seconds': time.perf_counter() - t0}

# Use ten seeds with fixed explanation and background samples to reduce
# variation unrelated to the repeated network training
N_SEEDS = 10
seeds = list(range(1, N_SEEDS + 1))
_rng = np.random.RandomState(0)
X_expl = X_test_arr[_rng.choice(X_test_arr.shape[0], size=min(500, X_test_arr.shape[0]), replace=False)]
bg = X_train_arr[_rng.choice(X_train_arr.shape[0], size=min(200, X_train_arr.shape[0]), replace=False)]

In [ ]:
# ============================================================
# RQ2 Multi-Seed Run: Train, Score, Explain Each Net per Seed
# ============================================================

# For each seed, train both frameworks and record the training diagnostics
# and per-feature SHAP importances. Ten matched seeds are used for the
# paired framework comparison described in the dissertation.
rows_train, shap_records = [], []
for s in seeds:
    # TensorFlow: train, score on the fixed test set, explain on the fixed sample
    m_tf, info_tf = train_tf_seed(X_train_arr, y_train_arr, s)
    auc_tf = roc_auc_score(y_test_arr, m_tf(X_test_arr, training=False).numpy().ravel())
    imp_tf = global_importance(as_2d(shap.GradientExplainer(m_tf, bg).shap_values(X_expl)))
    rows_train.append({'seed': s, 'framework': 'TensorFlow', 'test_auc': auc_tf, **info_tf})
    for feat, val in imp_tf.items():
        shap_records.append({'seed': s, 'framework': 'TensorFlow', 'feature': feat, 'importance': val})

# PyTorch: repeat the same steps using the same fixed samples so that
# the two framework implementations are compared under closely matched conditions
    m_pt, info_pt = train_torch_seed(X_train_arr, y_train_arr, s)
    m_pt.eval()
    with torch.no_grad():
        auc_pt = roc_auc_score(y_test_arr, m_pt(torch.tensor(X_test_arr, dtype=torch.float32)).numpy().ravel())
    imp_pt = global_importance(as_2d(
        shap.GradientExplainer(m_pt, torch.tensor(bg, dtype=torch.float32))
        .shap_values(torch.tensor(X_expl, dtype=torch.float32))))
    rows_train.append({'seed': s, 'framework': 'PyTorch', 'test_auc': auc_pt, **info_pt})
    for feat, val in imp_pt.items():
        shap_records.append({'seed': s, 'framework': 'PyTorch', 'feature': feat, 'importance': val})

    print(f"seed {s} done")

train_df = pd.DataFrame(rows_train)
shap_df = pd.DataFrame(shap_records)
print("multi-seed run complete")

In [ ]:
# ============================================================
# RQ2 Preview and Export
# ============================================================
# Paired Wilcoxon signed-rank test (Wilcoxon, 1945), Bonferroni corrected

print("Training behaviour per framework (mean, std):")
print(train_df.groupby('framework')[['test_auc', 'epochs', 'final_val_loss', 'train_seconds']]
      .agg(['mean', 'std']).round(4))
print()

wide = shap_df.pivot_table(index=['feature', 'seed'], columns='framework', values='importance').reset_index()
feats = wide['feature'].unique()
res = []
for f in feats:
    sub = wide[wide['feature'] == f]
    try:
        _, p = wilcoxon(sub['TensorFlow'].values, sub['PyTorch'].values)
    except ValueError:
        p = 1.0
    res.append({'feature': f, 'p_value': p})
res_df = pd.DataFrame(res)
res_df['p_bonferroni'] = (res_df['p_value'] * len(feats)).clip(upper=1.0)
print(f"Features differing between frameworks after Bonferroni: {int((res_df['p_bonferroni'] < 0.05).sum())} of {len(feats)}")
print()

train_df.to_csv(f'{DATASET}_rq2_training.csv', index=False)
shap_df.to_csv(f'{DATASET}_rq2_shap.csv', index=False)
print(f"Saved {DATASET}_rq2_training.csv and {DATASET}_rq2_shap.csv")
from google.colab import files
files.download(f'{DATASET}_rq2_training.csv')
files.download(f'{DATASET}_rq2_shap.csv')

In [ ]:
# ============================================================
# Figure: Framework Comparison (RQ2)
# ============================================================

# Compare the TensorFlow and PyTorch SHAP feature importances side by side.
# The eight most important features are selected using the mean importance
# across the two frameworks.

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

# Mean SHAP importance for each original feature within each framework
means = (
    shap_df
    .groupby(['feature', 'framework'])['importance']
    .mean()
    .unstack()
)

# Use the mean importance across TensorFlow and PyTorch to rank features
means['scale'] = means[['TensorFlow', 'PyTorch']].mean(axis=1)

# Select the eight most important features, ordered highest to lowest
d = means.sort_values('scale', ascending=False).head(8)

# Tidy the labels: the raw Taiwan column names are codes rather than descriptions
tidy = {
    'PAY_1': 'Repayment status, most recent',
    'PAY_2': 'Repayment status, 1 month earlier',
    'PAY_3': 'Repayment status, 2 months earlier',
    'PAY_4': 'Repayment status, 3 months earlier',
    'PAY_5': 'Repayment status, 4 months earlier',
    'PAY_6': 'Repayment status, 5 months earlier',

    'PAY_AMT1': 'Payment amount, most recent',
    'PAY_AMT2': 'Payment amount, 1 month earlier',
    'PAY_AMT3': 'Payment amount, 2 months earlier',
    'PAY_AMT4': 'Payment amount, 3 months earlier',
    'PAY_AMT5': 'Payment amount, 4 months earlier',
    'PAY_AMT6': 'Payment amount, 5 months earlier',

    'BILL_AMT1': 'Bill amount, most recent',
    'BILL_AMT2': 'Bill amount, 1 month earlier',
    'BILL_AMT3': 'Bill amount, 2 months earlier',
    'BILL_AMT4': 'Bill amount, 3 months earlier',
    'BILL_AMT5': 'Bill amount, 4 months earlier',
    'BILL_AMT6': 'Bill amount, 5 months earlier',

    'LIMIT_BAL': 'Credit limit',
    'EDUCATION': 'Education',
    'MARRIAGE': 'Marital status',
    'AGE': 'Age',
    'SEX': 'Sex',
}

labels = [tidy.get(f, f) for f in d.index]

print("Top eight features:")
print(labels)

# Compare the overall attribution scale between the two frameworks
print("\nTotal attribution mass:")
print(f"  TensorFlow: {means['TensorFlow'].sum():.4f}")
print(f"  PyTorch:    {means['PyTorch'].sum():.4f}")
print(
    f"  ratio TF/PyT: "
    f"{means['TensorFlow'].sum() / means['PyTorch'].sum():.3f}"
)

rho, _ = spearmanr(
    means['TensorFlow'],
    means['PyTorch']
)

print(f"  Spearman between rankings: {rho:.3f}")

In [ ]:
# ============================================================
# Grouped Bar Chart
# ============================================================

TEAL = "#5B9C8F"
RED = "#B5556A"
GREY = "#4A4A4A"

fig, ax = plt.subplots(figsize=(8.0, 4.8))

y = np.arange(len(d))
h = 0.38

# TensorFlow and PyTorch bars
ax.barh(
    y - h / 2,
    d['TensorFlow'],
    height=h,
    color=TEAL,
    label='TensorFlow'
)

ax.barh(
    y + h / 2,
    d['PyTorch'],
    height=h,
    color=RED,
    label='PyTorch'
)

# Feature labels
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)

# d is sorted from highest to lowest importance.
# Invert the axis so the most important feature appears at the top.
ax.invert_yaxis()

ax.set_xlabel(
    "Mean absolute SHAP value",
    fontsize=10,
    color=GREY
)

ax.set_ylabel(
    "Feature",
    fontsize=10,
    color=GREY
)

ax.legend(
    frameon=False,
    fontsize=9,
    loc='lower right'
)

# Gridlines only on the value axis so bar lengths are easy to compare
ax.grid(
    axis='x',
    alpha=0.3,
    linewidth=0.6
)

ax.set_axisbelow(True)

# Remove unnecessary borders
for s in ['top', 'right', 'left']:
    ax.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Save Figure
# ============================================================

# Save the figure to a local images directory
OUT_DIR = 'images'
os.makedirs(OUT_DIR, exist_ok=True)

out = os.path.join(OUT_DIR, 'fig_framework_taiwan.png')
fig.savefig(out, dpi=220, bbox_inches='tight', facecolor='white')

print("Written:", out)

In [ ]:
from google.colab import files
files.download('taiwan_xai_importance.csv')

In [ ]:
import os
[f for f in os.listdir() if f.endswith('.csv')]

In [ ]:
# ============================================================
# References
# ============================================================

# Chawla, N.V., Bowyer, K.W., Hall, L.O. and Kegelmeyer, W.P. (2002) 'SMOTE:
#   synthetic minority over-sampling technique', Journal of Artificial
#   Intelligence Research, 16, pp. 321-357. doi: 10.1613/jair.953.
#
# Chen, T. and Guestrin, C. (2016) 'XGBoost: a scalable tree boosting system',
#   Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge
#   Discovery and Data Mining, pp. 785-794. doi: 10.1145/2939672.2939785.
#
# Efron, B. (1979) 'Bootstrap methods: another look at the jackknife',
#   The Annals of Statistics, 7(1), pp. 1-26.
#   doi: 10.1214/aos/1176344552.
#
# Equal Employment Opportunity Commission (1978) Uniform Guidelines on Employee
#   Selection Procedures. 29 C.F.R. Part 1607.
#
# Kendall, M.G. and Babington Smith, B. (1939) 'The problem of m rankings',
#   The Annals of Mathematical Statistics, 10(3), pp. 275-287.
#   doi: 10.1214/aoms/1177732186.
#
# Kingma, D.P. and Ba, J. (2015) 'Adam: a method for stochastic optimization',
#   3rd International Conference on Learning Representations (ICLR).
#
# Lundberg, S.M. and Lee, S.-I. (2017) 'A unified approach to interpreting
#   model predictions', Advances in Neural Information Processing Systems,
#   30, pp. 4765-4774.
#
# Lundberg, S.M. et al. (2020) 'From local explanations to global understanding
#   with explainable AI for trees', Nature Machine Intelligence, 2(1),
#   pp. 56-67. doi: 10.1038/s42256-019-0138-9.
#
# Ribeiro, M.T., Singh, S. and Guestrin, C. (2016) '"Why should I trust you?":
#   explaining the predictions of any classifier', Proceedings of the 22nd
#   ACM SIGKDD International Conference on Knowledge Discovery and Data Mining,
#   pp. 1135-1144. doi: 10.1145/2939672.2939778.
#
# Sundararajan, M., Taly, A. and Yan, Q. (2017) 'Axiomatic attribution for deep
#   networks', Proceedings of the 34th International Conference on Machine
#   Learning, 70, pp. 3319-3328.
#
# Wilcoxon, F. (1945) 'Individual comparisons by ranking methods',
#   Biometrics Bulletin, 1(6), pp. 80-83. doi: 10.2307/3001968.
#
# Yeh, I. (2009) Default of Credit Card Clients [Dataset].
#   UCI Machine Learning Repository. doi: 10.24432/C55S3H.
#
# Yeh, I.-C. and Lien, C.-H. (2009) 'The comparisons of data mining techniques
#   for the predictive accuracy of probability of default of credit card
#   clients', Expert Systems with Applications, 36(2), pp. 2473-2480.
#   doi: 10.1016/j.eswa.2007.12.020.